# 02 · Motor de texto con BETO

**Fase 3 — El núcleo del aporte. Corre en Google Colab con GPU.**

Usa las particiones ya armadas (`corpus_train/val/test.csv`), donde **val y test son 100% reales**.

Pasos: línea base clásica → ajuste fino de BETO (3–4 épocas, lr ~2e-5) → comparación.

> ⚠️ En Colab: **Entorno de ejecución → Cambiar tipo → GPU (T4)** antes de entrenar.

## 0. Preparación (solo en Colab)

In [ ]:
# 1) Instala dependencias (Colab ya trae torch):
# !pip -q install -U transformers accelerate

# 2) SUBE tus 3 particiones a Colab con el ícono de carpeta 📁 (izquierda → Subir):
#      corpus_train.csv, corpus_val.csv, corpus_test.csv
#    (están en tu PC en:  data/particiones/ )
print('Descomenta el pip y sube corpus_train.csv, corpus_val.csv y corpus_test.csv.')

In [ ]:
import torch
print('GPU disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('⚠️ Sin GPU: activa GPU en Colab (Entorno de ejecución → Cambiar tipo).')

## 1. Cargar las particiones (val y test = reales)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)

def buscar(nombre):
    for p in [Path(nombre), Path('data/particiones') / nombre,
              Path.cwd().parent / 'data' / 'particiones' / nombre]:
        if p.exists():
            return p
    return None

p_tr, p_va, p_te = buscar('corpus_train.csv'), buscar('corpus_val.csv'), buscar('corpus_test.csv')

if p_tr is not None and p_va is not None:
    train = pd.read_csv(p_tr)[['texto', 'etiqueta']].dropna()
    val = pd.read_csv(p_va)[['texto', 'etiqueta']].dropna()
    test = pd.read_csv(p_te)[['texto', 'etiqueta']].dropna() if p_te is not None else None
    print('Particiones reales cargadas ✔')
else:
    print('No se encontraron particiones -> DEMO en español (placeholder) para probar el flujo.')
    from sklearn.model_selection import train_test_split
    fraude = ['Su cuenta sera bloqueada, ingrese aqui para validar sus datos',
              'Felicidades gano un premio, confirme sus datos para cobrar',
              'Alerta: movimiento sospechoso, verifique su clave ahora',
              'Su paquete esta retenido, pague la aduana en este enlace'] * 15
    legit = ['Hola, te confirmo la reunion de manana a las diez',
             'Recordatorio de tu cita medica el viernes',
             'Gracias por tu compra, tu pedido llega el lunes',
             'Tu factura del mes ya esta disponible en la app'] * 15
    df = pd.DataFrame({'texto': fraude + legit,
                       'etiqueta': ['fraudulento']*len(fraude) + ['legitimo']*len(legit)})
    df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)
    train, resto = train_test_split(df, test_size=0.30, stratify=df['etiqueta'], random_state=SEED)
    val, test = train_test_split(resto, test_size=0.50, stratify=resto['etiqueta'], random_state=SEED)

print(f'train={len(train)}  val={len(val)}  test={len(test) if test is not None else 0}')
print('Train:', train['etiqueta'].value_counts().to_dict())
print('Val  :', val['etiqueta'].value_counts().to_dict())

## 2. Línea base clásica en español (el rival)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report, f1_score

base = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2), min_df=1),
    LogisticRegression(max_iter=1000, class_weight='balanced'),
)
base.fit(train['texto'], train['etiqueta'])
pred_base = base.predict(val['texto'])
f1_base = f1_score(val['etiqueta'], pred_base, pos_label='fraudulento', zero_division=0)
print(f'Baseline clásico — F1 (fraudulento) = {f1_base:.3f}')
print(classification_report(val['etiqueta'], pred_base, digits=3, zero_division=0))

## 3. Ajuste fino de BETO

In [ ]:
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer)
from torch.utils.data import Dataset
from collections import Counter

CKPT = 'dccuchile/bert-base-spanish-wwm-cased'  # BETO
ETIQUETAS = ['legitimo', 'fraudulento']
lab2id = {l: i for i, l in enumerate(ETIQUETAS)}

tok = AutoTokenizer.from_pretrained(CKPT)

class DatasetTexto(Dataset):
    def __init__(self, textos, etiquetas):
        self.enc = tok(list(textos), truncation=True, padding=True, max_length=128)
        self.labels = [lab2id[e] for e in etiquetas]
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
        item['labels'] = torch.tensor(self.labels[i])
        return item

ds_tr = DatasetTexto(train['texto'], train['etiqueta'])
ds_val = DatasetTexto(val['texto'], val['etiqueta'])

In [ ]:
# Pesos por clase para la pérdida (ponderar en vez de descartar)
cnt = Counter(train['etiqueta'])
total = sum(cnt.values())
pesos = torch.tensor([total / (2 * cnt[l]) for l in ETIQUETAS], dtype=torch.float)
print('Conteo por clase:', dict(cnt))

class TrainerPonderado(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        loss_fct = torch.nn.CrossEntropyLoss(weight=pesos.to(model.device))
        loss = loss_fct(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def metricas(pred):
    logits = pred.predictions[0] if isinstance(pred.predictions, tuple) else pred.predictions
    labels = pred.label_ids
    preds = np.argmax(logits, axis=1)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, labels=[0, 1],
                                                  average=None, zero_division=0)
    return {'accuracy': accuracy_score(labels, preds),
            'f1_fraudulento': f1[1],
            'recall_fraudulento': r[1],
            'precision_fraudulento': p[1]}

In [ ]:
modelo = AutoModelForSequenceClassification.from_pretrained(
    CKPT, num_labels=2, id2label={i: l for l, i in lab2id.items()}, label2id=lab2id)

args = TrainingArguments(
    output_dir='beto_ckpt',
    num_train_epochs=4,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy='epoch',        # en transformers viejos: evaluation_strategy
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_fraudulento',
    seed=SEED,
    logging_steps=20,
    report_to='none',
)

trainer = TrainerPonderado(model=modelo, args=args,
                           train_dataset=ds_tr, eval_dataset=ds_val,
                           compute_metrics=metricas)
trainer.train()

## 4. Evaluación y comparación con la línea base

In [ ]:
res = trainer.evaluate()
f1_beto = res['eval_f1_fraudulento']
print(f'BETO     — F1 (fraudulento) = {f1_beto:.3f}')
print(f'Baseline — F1 (fraudulento) = {f1_base:.3f}')
print('\nBETO gana al baseline' if f1_beto > f1_base else '\nRevisar: el baseline no fue superado')
print('(El veredicto formal se hace con McNemar sobre el TEST en 03_evaluacion.)')

In [ ]:
# Guardar el modelo ajustado y las predicciones de validación
trainer.save_model('beto_final')
tok.save_pretrained('beto_final')

logits = trainer.predict(ds_val).predictions
logits = logits[0] if isinstance(logits, tuple) else logits
pred_beto = [ETIQUETAS[i] for i in np.argmax(logits, axis=1)]
pd.DataFrame({'texto': val['texto'].values, 'y_real': val['etiqueta'].values,
              'pred_beto': pred_beto, 'pred_baseline': pred_base}).to_csv(
    '02_texto_val_pred.csv', index=False)
print('Modelo en beto_final/ y predicciones en 02_texto_val_pred.csv ✔')
print('⚠️ En Colab: descarga la carpeta beto_final o cópiala a Google Drive antes de cerrar.')

## Qué sigue

- El **test** sigue sin abrirse: se usa en `03_evaluacion.ipynb` (McNemar entre BETO y baseline, criterio F1 ≥ 0.90).
- Si BETO sobreajusta (F1 alto en train, bajo en val): reduce épocas, sube el peso de la clase minoritaria o agrega más datos reales.